In [ ]:
# If needed:
# !pip install ultralytics torchmetrics opencv-python

import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from tqdm import tqdm

from ultralytics import YOLO
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [ ]:
# Configuration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Update these paths
XML_PATH = "path/to/MVI_20011.xml"
IMG_DIR = "path/to/MVI_20011"
YOLO_WEIGHTS = "path/to/best.pt"   # your YOLO trained weights

BATCH_SIZE = 8
NUM_WORKERS = 2
CONF_THRESH = 0.30
IOU_MATCH_THRESH = 0.50

# UA-DETRAC mapping you are using in GT
CLASS_TO_ID = {"car": 1, "van": 2, "bus": 3, "others": 4}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

# IMPORTANT:
# If YOLO predicts classes as 0..3 and GT is 1..4 -> add offset +1
YOLO_LABEL_OFFSET = 1

In [ ]:
# Parse UA-DETRAC XML ground truth

def parse_detrac_xml(xml_path: str):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    annotations = {}
    for frame in root.findall(".//frame"):
        fid = int(frame.get("num"))
        target_list = frame.find("target_list")

        boxes, labels = [], []
        if target_list is not None:
            for target in target_list.findall("target"):
                attr = target.find("attribute")
                box = target.find("box")
                if attr is None or box is None:
                    continue

                vtype = attr.get("vehicle_type", "others").lower()
                if vtype not in CLASS_TO_ID:
                    vtype = "others"

                left = float(box.get("left", 0))
                top = float(box.get("top", 0))
                width = float(box.get("width", 0))
                height = float(box.get("height", 0))

                x1, y1 = left, top
                x2, y2 = left + width, top + height
                if x2 <= x1 or y2 <= y1:
                    continue

                boxes.append([x1, y1, x2, y2])
                labels.append(CLASS_TO_ID[vtype])

        if boxes:
            annotations[fid] = {
                "boxes": torch.tensor(boxes, dtype=torch.float32),
                "labels": torch.tensor(labels, dtype=torch.int64),
            }

    return annotations

In [ ]:
# =========================
# 4) Dataset
# =========================
class DetracEvalDataset(Dataset):
    def __init__(self, img_dir, annotations):
        self.img_dir = Path(img_dir)
        self.annotations = annotations
        self.frame_ids = sorted(list(annotations.keys()))
        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx):
        fid = self.frame_ids[idx]
        img_path = self.img_dir / f"img{fid:05d}.jpg"

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(f"Missing image: {img_path}")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_tensor = self.to_tensor(img_rgb)

        target = {
            "boxes": self.annotations[fid]["boxes"].clone(),
            "labels": self.annotations[fid]["labels"].clone(),
            "image_id": torch.tensor([fid], dtype=torch.int64),
            "img_path": str(img_path),  # useful for YOLO batch inference
        }
        return img_tensor, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

In [ ]:
# =========================
# 5) IoU + P/R/F1 helpers

def box_iou_xyxy(boxes1, boxes2):
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)

    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)

    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]

    union = area1[:, None] + area2 - inter
    return inter / union.clamp(min=1e-6)

@torch.no_grad()
def compute_precision_recall_f1(preds, gts, iou_thresh=0.5, score_thresh=0.3):
    TP, FP, FN = 0, 0, 0

    for pred, gt in zip(preds, gts):
        p_boxes = pred["boxes"]
        p_scores = pred["scores"]
        p_labels = pred["labels"]

        g_boxes = gt["boxes"]
        g_labels = gt["labels"]

        keep = p_scores >= score_thresh
        p_boxes, p_scores, p_labels = p_boxes[keep], p_scores[keep], p_labels[keep]

        classes = torch.unique(torch.cat([p_labels, g_labels], dim=0)) if (len(p_labels) or len(g_labels)) else torch.tensor([])
        for c in classes.tolist():
            pb = p_boxes[p_labels == c]
            ps = p_scores[p_labels == c]
            gb = g_boxes[g_labels == c]

            if len(pb) == 0 and len(gb) == 0:
                continue
            if len(pb) == 0:
                FN += len(gb)
                continue
            if len(gb) == 0:
                FP += len(pb)
                continue

            order = torch.argsort(ps, descending=True)
            pb = pb[order]
            ious = box_iou_xyxy(pb, gb)

            matched_gt = torch.zeros(len(gb), dtype=torch.bool)
            for i in range(len(pb)):
                best_iou, best_j = torch.max(ious[i], dim=0)
                if best_iou >= iou_thresh and not matched_gt[best_j]:
                    TP += 1
                    matched_gt[best_j] = True
                else:
                    FP += 1

            FN += (~matched_gt).sum().item()

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {"TP": TP, "FP": FP, "FN": FN, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# YOLO specific evaluation loop

@torch.no_grad()
def evaluate_yolo(model, dataloader, conf_thresh=0.3, iou_match_thresh=0.5, yolo_label_offset=1):
    metric_map = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True)

    all_preds = []
    all_gts = []

    for _, targets in tqdm(dataloader, desc="Evaluating YOLO"):
        # YOLO can infer from file paths directly
        img_paths = [t["img_path"] for t in targets]
        results = model.predict(
            source=img_paths,
            conf=conf_thresh,
            iou=0.7,            # NMS IoU, separate from eval IoU
            verbose=False,
            device=0 if DEVICE == "cuda" else "cpu"
        )

        preds_batch = []
        gts_batch = []

        for r, tgt in zip(results, targets):
            boxes = r.boxes.xyxy.cpu() if r.boxes is not None else torch.empty((0, 4), dtype=torch.float32)
            scores = r.boxes.conf.cpu() if r.boxes is not None else torch.empty((0,), dtype=torch.float32)
            labels = r.boxes.cls.cpu().long() + yolo_label_offset if r.boxes is not None else torch.empty((0,), dtype=torch.int64)

            pred = {"boxes": boxes, "scores": scores, "labels": labels}
            gt = {"boxes": tgt["boxes"].cpu(), "labels": tgt["labels"].cpu()}

            preds_batch.append(pred)
            gts_batch.append(gt)

        metric_map.update(preds_batch, gts_batch)
        all_preds.extend(preds_batch)
        all_gts.extend(gts_batch)

    map_result = metric_map.compute()
    prf1_result = compute_precision_recall_f1(
        all_preds, all_gts,
        iou_thresh=iou_match_thresh,
        score_thresh=conf_thresh
    )
    return map_result, prf1_result

In [ ]:
# Run Evaluation

annotations = parse_detrac_xml(XML_PATH)
dataset = DetracEvalDataset(IMG_DIR, annotations)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

model = YOLO(YOLO_WEIGHTS)

map_result, prf1_result = evaluate_yolo(
    model,
    loader,
    conf_thresh=CONF_THRESH,
    iou_match_thresh=IOU_MATCH_THRESH,
    yolo_label_offset=YOLO_LABEL_OFFSET
)

print("===== Precision / Recall / F1 =====")
print(f"Precision: {prf1_result['precision']:.4f}")
print(f"Recall:    {prf1_result['recall']:.4f}")
print(f"F1-score:  {prf1_result['f1']:.4f}")
print(f"TP/FP/FN:  {prf1_result['TP']}/{prf1_result['FP']}/{prf1_result['FN']}")

print("\n===== mAP =====")
print(f"mAP@[0.50:0.95]: {map_result['map'].item():.4f}")
print(f"mAP@0.50:        {map_result['map_50'].item():.4f}")
print(f"mAP@0.75:        {map_result['map_75'].item():.4f}")
print(f"mAR@100:         {map_result['mar_100'].item():.4f}")

if map_result.get("map_per_class", None) is not None:
    print("\nClass-wise AP:")
    classes = map_result["classes"].cpu().tolist()
    aps = map_result["map_per_class"].cpu().tolist()
    for cid, ap in zip(classes, aps):
        print(f"{ID_TO_CLASS.get(int(cid), f'class_{int(cid)}')}: {ap:.4f}")